# NFT参照コードの使用例：各機能の確認

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

動作確認に使用するユーザをロードします。

In [2]:
var users = [];
for(var i=0; i<=5; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct
user3: u58185320 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
user4: u10676071 eEYRycGm4znXxnf7bTngX72JmYk57r
user5: u00571239 eWW6LCfRSpoVuudzZHa4ZwCjkkizov


ユーザ名の逆引き関数を準備します。

In [3]:
var userids = users.map(u => u.id);
function nameOfUserId(id){
    var i = userids.indexOf(id);
    if(i >= 0) return 'user'+i;
    return 'user_not_found';
}

## NFTコレクションのデプロイ

NFTコントラクトのサンプルコードを読み出します。

In [4]:
var { default: func } = await import('./nft100.mjs');
var m = /function.*\{([\s\S]*)\}/.exec(func.toString());
var code = m[1];

NFTコレクションの名称と略号を書き換えます。

In [5]:
var code = code.replace(/NFT_NAME = '.*'/, `NFT_NAME = 'sample NFT collection #123'`);
var code = code.replace(/NFT_SYMBOL = '.*'/, `NFT_SYMBOL = 'SMP123'`);

NFTコレクションの発行数を10とし、発行元をユーザ0に書き換えます。  
初期状態ではユーザ0が全トークンのオーナになります。

In [6]:
var code = code.replace(/TOTAL_SUPPLY = \d+/, `TOTAL_SUPPLY = 10`);
var code = code.replace(/INITIAL_OWNER = '.*'/, `INITIAL_OWNER = '${users[0].id}'`);

スマートコントラクトをデプロイします。  
変数nftidにNFTコントラクトのIDが格納されます。

In [7]:
var func = new Function(code);
var nftid = await deploySmartContract({func:'string', args:'any'}, func, { name: 'nft100' });

NFTコントラクトのアクセス範囲を、動作確認に使用するユーザに設定します。

In [8]:
await rpc.call(adminWallet, 'c1update', {id:nftid, prop:'accessible_to', value: userids});

{
  txno: 702186,
  txid: 'xcGoDNNv42nc2TFbA2Ms9GCRWPT5YHbNC9jy6F9oLhX7AB',
  status: 'ok',
  value: null
}

## NFTコントラクトの各インタフェースの動作確認

### name
NFTコレクションの名称を取得します。

In [9]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'name'});
console.log('name:', resp);

name: {
  txno: 702187,
  txid: 'xcbxYUTznZvCgNSpPKcw5fJpeH3HtaTECjQGgWtS6aSax',
  status: 'ok',
  value: 'sample NFT collection #123'
}


### symbol
NFTコレクションの略号を取得します。

In [10]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'symbol'});
console.log('symbol:', resp);

symbol: {
  txno: 702188,
  txid: 'x2ABdkrnXD7fnBDxeUrZ9ZkashtukZ8fMEqV5tLnvrKhJB',
  status: 'ok',
  value: 'SMP123'
}


### totalSupply
NFTコレクションに含まれるトークンの総数を取得します。

In [11]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'totalSupply'});
console.log('totalSupply:', resp);

totalSupply: {
  txno: 702189,
  txid: 'x4jpsVoLQgPnkAJqtzzfhMicPNmgtU5kSFiKqVGnn89YEB',
  status: 'ok',
  value: 10
}


### balanceOf
指定したオーナが現在保持するトークンの数を取得します。

In [12]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[0].id}});
console.log('balanceOf(user0):', resp);

balanceOf(user0): {
  txno: 702190,
  txid: 'xPiFhtChqHtjaAP7feorwxhvippd8sUpbkmccsXd58Yaz',
  status: 'ok',
  value: 10
}


In [13]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', resp);

balanceOf(user1): {
  txno: 702191,
  txid: 'x8RGzV8u4PZFuKnWQ6AdMhPWLWNUNGv7E5znWWoFvDY5EB',
  status: 'ok',
  value: 0
}


### ownerOf
指定したトークンの現在のオーナを取得します。

In [14]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token0'}});
console.log('ownerOf(token0):', resp, nameOfUserId(resp.value));

ownerOf(token0): {
  txno: 702192,
  txid: 'xYyA97prY9TwzVS4maFNUJhkXTv3By9ui8ZPjymy3FUYCB',
  status: 'ok',
  value: 'u73621973'
} user0


### tokenURI
指定したトークンに関するメタデータのURIを取得します。

In [15]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenURI', args: {tokenId: 'token0'}});
console.log('tokenURI(token0):', resp);

tokenURI(token0): {
  txno: 702193,
  txid: 'xzKwVnEQ9cfeSeri4jx5JyW9TLMnnEnoFXJQWWhHdB9EJB',
  status: 'ok',
  value: 'https://anywhere.com/token0'
}


### tokenByIndex
NFTコレクションに含まれる全トークンのうち、インデックスで指定されるトークンを取得します。

In [16]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenByIndex', args: {index: 8}});
console.log('tokenByIndex(8):', resp);

tokenByIndex(8): {
  txno: 702194,
  txid: 'xtDZby3cXU5e2aVssxiHx7z3UXgMMncfyVewRrLGer4THB',
  status: 'ok',
  value: 'token8'
}


### tokenOfOwnerByIndex
指定されたオーナが保持するトークンのうち、インデックスで指定されたトークンを取得します。

In [17]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenOfOwnerByIndex', args: {owner: users[0].id, index: 8}});
console.log('tokenOfOwnerByIndex(8):', resp);

tokenOfOwnerByIndex(8): {
  txno: 702195,
  txid: 'xTsVrzQPzY4Cbf3kdKwjjUsMR4wBeHWgqNttLwYbHD4qx',
  status: 'ok',
  value: 'token8'
}


### transferFrom
指定したトークンを別のオーナに転送します。（オーナが変わります）  
token0をuser0からuser1へ転送します。

In [18]:
var resp = await rpc.call(users[0].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[1].id,
      tokenId: 'token0'
    }
});
console.log('transferFrom(user0,user1,token0):', resp);

transferFrom(user0,user1,token0): {
  txno: 702196,
  txid: 'x3d6NkL3pKMp5J8yEeeE3suZhHbASoBJ3XmbUo26J2yW7',
  status: 'ok',
  value: null
}


転送後の保持数を確認します。

In [19]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'balanceOf', args: {owner: users[0].id}});
console.log('balanceOf(user0):', resp.value);
var resp = await rpc.call(users[0].wallet, nftid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', resp.value);

balanceOf(user0): 9
balanceOf(user1): 1


転送後のオーナを確認します。

In [20]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token0'}});
console.log('ownerOf(token0):', resp, nameOfUserId(resp.value));

ownerOf(token0): {
  txno: 702199,
  txid: 'xMqgmJKF7m4WYBrWbMxqfDtBghrccak77fe98PX34EZp6',
  status: 'ok',
  value: 'u28169743'
} user1


user2はオーナではなく、ここで転送しようとするとエラーになります。

In [21]:
var resp = await rpc.call(users[2].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[1].id,
      tokenId: 'token1'
    }
});
console.log('transferFrom(user0,user1,token2):', resp);

transferFrom(user0,user1,token2): {
  txno: 702200,
  txid: 'xrSSjqjX36XVVuX8QdsY5C7icHubdbJsCTzEec7tedyCKB',
  status: 'thrown',
  value: 'caller not authorized for token1'
}


### approve
トークンを転送する権限を委任します。

token2に関して、user0からの転送の権限を、user2に委任します。

In [22]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'approve', args: {approved: users[2].id, tokenId: 'token2'}});
console.log('approve(user2,token2):', resp);

approve(user2,token2): {
  txno: 702201,
  txid: 'xDxQ9UBnFskExrSVaivJ4v9c8aHS46fRN4XvVEzQA2kD8',
  status: 'ok',
  value: null
}


### getApproved
転送権限の委任先を取得します。

In [23]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'getApproved', args: {tokenId: 'token2'}});
console.log('getApproved(token2):', resp, nameOfUserId(resp.value));

getApproved(token2): {
  txno: 702202,
  txid: 'xLuiSzVvvTHWbrnNGWcdMnm8mBYE4t4Yvy4R8gBcAjm7KB',
  status: 'ok',
  value: 'u53371386'
} user2


委任されていない場合は、nullが返ります。

In [24]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'getApproved', args: {tokenId: 'token3'}});
console.log('getApproved(token3):', resp.value);

getApproved(token3): null


### approveの効果
転送権限の委任により、委任先の操作によるトークンの転送が可能になります。  
token2に関して、user0からuser1への転送をuser2が実行します。

In [25]:
var resp = await rpc.call(users[2].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[1].id,
      tokenId: 'token2'
    }
});
console.log('transferFrom(user0,user1,token2):', resp);

transferFrom(user0,user1,token2): {
  txno: 702204,
  txid: 'xUdq2Lfe3mycdZV5zJJjpuCjBkminWmRdFYmpHqcHBjrLB',
  status: 'ok',
  value: null
}


転送後のオーナを確認します。

In [26]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token2'}});
console.log('ownerOf(token2):', resp.value, nameOfUserId(resp.value));

ownerOf(token2): u28169743 user1


転送後は、転送権限の委任はクリアされています。

In [27]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'getApproved', args: {tokenId: 'token2'}});
console.log('getApproved(token2):', resp.value);

getApproved(token2): null


### setApprovalForAll
操作権限を委任または解除します。

user0の操作権限をuser3に委任します。

In [28]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'setApprovalForAll', args: {operator: users[3].id, approved: true }});
console.log('setApprovedForAll(user3,true):', resp);

setApprovedForAll(user3,true): {
  txno: 702207,
  txid: 'x7UyjMBvbfKKwCfw44rTjZWxSLhRSothzT885Xt9Lqvjy',
  status: 'ok',
  value: null
}


### isApprovedForAll
指定したオーナの操作権限について、指定したオペレータへの委任の有無を取得します。

In [29]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'isApprovedForAll', args: {owner: users[0].id, operator: users[3].id}});
console.log('isApprovedForAll(user0,user3):', resp);

isApprovedForAll(user0,user3): {
  txno: 702208,
  txid: 'xrVEEDgtheFAQQte2beFz7gmDXBMWUbSRfTMNX9ye4675',
  status: 'ok',
  value: true
}


### setApprovalForAllの効果
- 全てのトークンに関して、委任元からの転送を委任先が実行できます。
- 全てのトークンに関して、委任元からのapproveを委任先が実行できます。

token4に関して、user0からuser1への転送をuser3が実行します。

In [30]:
var resp = await rpc.call(users[3].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[1].id,
      tokenId: 'token4'
    }
});
console.log('transferFrom(user0,user1,token4):', resp);

transferFrom(user0,user1,token4): {
  txno: 702209,
  txid: 'xriGLhzhxex5YuHjdXVJwsnYwo6hQHK9oALJhKaNvEVX3',
  status: 'ok',
  value: null
}


転送後のオーナを確認します。

In [31]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token4'}});
console.log('ownerOf(token4):', resp.value, nameOfUserId(resp.value));

ownerOf(token4): u28169743 user1


token5に関して、user4へのapproveをuser3が実行します。  
(token5のオーナがuser0であり、user0からuser3へsetApprovalForAllがされているから）

In [32]:
var resp = await rpc.call(users[3].wallet, nftid, {func: 'approve', args: {approved: users[4].id, tokenId: 'token5'}});
console.log('approve(user4,token5):', resp);

approve(user4,token5): {
  txno: 702211,
  txid: 'x8sMz3msyHQ9yeuVB2y7AyYh6gwNXMEWig2nECzbcQKCs',
  status: 'ok',
  value: null
}


getApprovedで確認します。

In [33]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'getApproved', args: {tokenId: 'token5'}});
console.log('getApproved(token5):', resp.value, nameOfUserId(resp.value));

getApproved(token5): u10676071 user4


### safeTransferFrom
transferFromと同様にトークンを転送しますが、  
トークンの転送先がコントラクトの場合、転送先がonNFTReceivedに応答することが条件になります。

転送先となるテスト用のコントラクトをデプロイします。  
変数cidに作成したコントラクトのIDが格納されます。  
このコントラクトは、受け取ったトークンをユーザ5に自動的に転送します。

In [34]:
var cid = await deploySmartContract({func:'string', args:'any'}, function nft_receiver(func, args) {
  if(func=== 'onNFTReceived'){ // args:{operator, from, tokenId, data}
      openContract('__nftid__').call({func: 'safeTransferFrom', args: { from: getContractId(), to: '__userid5__', tokenId: args.tokenId }});
      return 'd56Kq6n1'; // successfully processed
  }
}, {replacers: [[/__nftid__/, nftid], [/__userid5__/, users[5].id]]});

作成したコントラクトがonNFTReceivedを受け付けられるように、accessible_toを設定します。

In [35]:
await rpc.call(adminWallet, 'c1update', {id:cid, prop:'accessible_to', value: [nftid]});

{
  txno: 702216,
  txid: 'xCLewjMa7rGkFTUbEmqCuXq3sNp4tVioDo3Lmw7zHtPUv',
  status: 'ok',
  value: null
}

作成したコントラクトがNFTコントラクトにアクセスできるように、accessible_toを設定します。

In [36]:
await rpc.call(adminWallet, 'c1update', {id:nftid, prop:'add accessible_to', value: [cid]});

{
  txno: 702217,
  txid: 'xKYGevnapjfrbeQn7kVkzT4bs5FsraBucwo86sTzhdZzs',
  status: 'ok',
  value: null
}

user0からcidへtoken6を転送します。

In [37]:
var resp = await rpc.call(users[0].wallet, nftid, {func: "safeTransferFrom",
    args: {
      from: users[0].id,
      to: cid,
      tokenId: 'token6'
    }
});
console.log('transferFrom(user0,cid,token6):', resp);

transferFrom(user0,cid,token6): {
  txno: 702218,
  txid: 'xu6MbsMJufkeAK5C4gneCf92QaN2iyATSHn6k4Brzk63LB',
  status: 'ok',
  value: null
}


転送後のオーナを確認します。

In [38]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token6'}});
console.log('ownerOf(token6):', resp.value, nameOfUserId(resp.value));

ownerOf(token6): u00571239 user5


間違えてNFTコントラクトにsafeTransferFromした場合には、エラーになります。(onNFTReceivedが処理されないため)

In [39]:
var resp = await rpc.call(users[0].wallet, nftid, {func: "safeTransferFrom",
    args: {
      from: users[0].id,
      to: nftid,
      tokenId: 'token7'
    }
});
console.log('transferFrom(user0,nftid,token7):', resp);

transferFrom(user0,nftid,token7): {
  txno: 702222,
  txid: 'xsv3BeGfALpgbK2xB8HeL36MLXJnxLdu6nPnu2YT97o7PB',
  status: 'aborted',
  value: 'denied: accessible permission\n' +
    '    at checkOnNFTReceived (c041411546:163:16)\n' +
    '    at c041411546:129:17'
}


### burn
burnはインタフェースとしては実装していませんが、   
例えばc0000に転送することで、トークンを破棄する効果が期待できます。

In [40]:
var resp = await rpc.call(users[0].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: 'c0000',
      tokenId: 'token8'
    }
});
console.log('transferFrom(user0,c0000,token8):', resp);

transferFrom(user0,c0000,token8): {
  txno: 702224,
  txid: 'xgSMA6FYN46ooWMdafi29oxYjoUYX8xnnKQao5MvCzhNCB',
  status: 'ok',
  value: null
}


転送後のオーナを確認します。

In [41]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token8'}});
console.log('ownerOf(token8):', resp.value);

ownerOf(token8): c0000


### トークンの保持状況の確認

In [42]:
for(var i=0; i<=5; i++){
    var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[i].id}});
    var tokens = [];
    var n = resp.value;
    for(var j=0; j<n; j++){
        var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenOfOwnerByIndex', args: {owner: users[i].id, index: j}});
        tokens[j] = resp.value;
    }
    console.log(`tokens of user${i}:`, tokens);
}

tokens of user0: [ 'token9', 'token1', 'token5', 'token3', 'token7' ]
tokens of user1: [ 'token0', 'token2', 'token4' ]
tokens of user2: []
tokens of user3: []
tokens of user4: []
tokens of user5: [ 'token6' ]


各トークンのownerも確認します

In [43]:
for(var i=0; i<10; i++){
    var resp = await rpc.call(users[5].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token'+i}});
    console.log(`owner of token${i}:`, resp.value, nameOfUserId(resp.value));
}

owner of token0: u28169743 user1
owner of token1: u73621973 user0
owner of token2: u28169743 user1
owner of token3: u73621973 user0
owner of token4: u28169743 user1
owner of token5: u73621973 user0
owner of token6: u00571239 user5
owner of token7: u73621973 user0
owner of token8: c0000 user_not_found
owner of token9: u73621973 user0
